**Problema de Negócio:** Transportes de carnes/vacinas exigem temperatura constante (SLA de -15°C). Sensores IoT nos caminhões disparam a cada segundo, mas frequentemente enviam "falsos positivos" (picos irreais de 50°C por 1 segundo devido a falhas elétricas). 

**Objetivo:** Limpar os erros do sensor e identificar quais caminhões tiveram falhas reais de refrigeração que ultrapassaram 15 minutos, invalidando o lote da carga.

In [0]:
%python
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

tempos = [datetime(2023, 10, 1, 8, 0, 0) + timedelta(seconds=i) for i in range(3600)]

temp_a = np.random.normal(-18, 0.5, 3600)
temp_a[500] = 45.0  # Erro absurdo 1
temp_a[1500] = -85.0 # Erro absurdo 2

temp_b = np.random.normal(-18, 0.5, 3600)
temp_b[1200:] = temp_b[1200:] + np.linspace(0, 15, 2400) # Temperatura sobe gradualmente para -3°C

df_a = pd.DataFrame({'truck_id': 'TRUCK-A', 'timestamp': tempos, 'temperatura_c': temp_a})
df_b = pd.DataFrame({'truck_id': 'TRUCK-B', 'timestamp': tempos, 'temperatura_c': temp_b})

df_raw = pd.concat([df_a, df_b])

spark_df_bronze = spark.createDataFrame(df_raw)
spark_df_bronze.createOrReplaceTempView("bronze_iot_sensors")

display(spark_df_bronze)

truck_id,timestamp,temperatura_c
TRUCK-A,2023-10-01T08:00:00.000Z,-18.126362171261064
TRUCK-A,2023-10-01T08:00:01.000Z,-17.724022058954684
TRUCK-A,2023-10-01T08:00:02.000Z,-18.209010594843214
TRUCK-A,2023-10-01T08:00:03.000Z,-18.355261919081094
TRUCK-A,2023-10-01T08:00:04.000Z,-18.27579189270581
TRUCK-A,2023-10-01T08:00:05.000Z,-17.812750351751212
TRUCK-A,2023-10-01T08:00:06.000Z,-17.895031592062733
TRUCK-A,2023-10-01T08:00:07.000Z,-17.57088548254289
TRUCK-A,2023-10-01T08:00:08.000Z,-18.714856248694034
TRUCK-A,2023-10-01T08:00:09.000Z,-17.315814163162315


In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql import functions as F

df_bronze = spark.table("bronze_iot_sensors")

w = Window.partitionBy("truck_id").orderBy(F.col("timestamp").cast("long")).rangeBetween(-30, 30)

df_silver = df_bronze.withColumn("temp_suavizada", F.avg("temperatura_c").over(w))

df_silver = df_silver.filter((F.col("temperatura_c") > -50) & (F.col("temperatura_c") < 50))

df_silver.createOrReplaceTempView("silver_cold_chain")

display(df_silver)

truck_id,timestamp,temperatura_c,temp_suavizada
TRUCK-A,2023-10-01T08:00:00.000Z,-18.126362171261064,-17.98210928210468
TRUCK-A,2023-10-01T08:00:01.000Z,-17.724022058954684,-17.98757666140544
TRUCK-A,2023-10-01T08:00:02.000Z,-18.209010594843214,-17.997428209442816
TRUCK-A,2023-10-01T08:00:03.000Z,-18.355261919081094,-17.989488896523856
TRUCK-A,2023-10-01T08:00:04.000Z,-18.27579189270581,-17.978262968740346
TRUCK-A,2023-10-01T08:00:05.000Z,-17.812750351751212,-17.978622277790038
TRUCK-A,2023-10-01T08:00:06.000Z,-17.895031592062733,-17.995766267595535
TRUCK-A,2023-10-01T08:00:07.000Z,-17.57088548254289,-18.00470357561517
TRUCK-A,2023-10-01T08:00:08.000Z,-18.714856248694034,-17.983722491915984
TRUCK-A,2023-10-01T08:00:09.000Z,-17.315814163162315,-17.987039152468984


In [0]:
%sql
WITH Calculo_Tempo AS (
    SELECT 
        truck_id,
        COUNT(*) AS segundos_fora_do_padrao
    FROM silver_cold_chain
    WHERE temp_suavizada > -15.0 
    GROUP BY truck_id
)

SELECT 
    truck_id,
    ROUND(segundos_fora_do_padrao / 60.0, 1) AS minutos_fora_do_padrao,
    CASE 
        WHEN (segundos_fora_do_padrao / 60.0) > 15 THEN '🚨 LOTE INVALIDADO (Risco Sanitário)'
        ELSE '✅ LOTE APROVADO'
    END AS status_qualidade
FROM Calculo_Tempo
ORDER BY minutos_fora_do_padrao DESC;

truck_id,minutos_fora_do_padrao,status_qualidade
TRUCK-B,32.2,🚨 LOTE INVALIDADO (Risco Sanitário)


In [0]:
%sql
SELECT timestamp, truck_id, temp_suavizada 
FROM silver_cold_chain 
WHERE minute(timestamp) % 2 = 0 

timestamp,truck_id,temp_suavizada
2023-10-01T08:00:00.000Z,TRUCK-A,-17.98210928210468
2023-10-01T08:00:01.000Z,TRUCK-A,-17.98757666140544
2023-10-01T08:00:02.000Z,TRUCK-A,-17.997428209442816
2023-10-01T08:00:03.000Z,TRUCK-A,-17.989488896523856
2023-10-01T08:00:04.000Z,TRUCK-A,-17.978262968740346
2023-10-01T08:00:05.000Z,TRUCK-A,-17.978622277790038
2023-10-01T08:00:06.000Z,TRUCK-A,-17.995766267595535
2023-10-01T08:00:07.000Z,TRUCK-A,-18.00470357561517
2023-10-01T08:00:08.000Z,TRUCK-A,-17.983722491915984
2023-10-01T08:00:09.000Z,TRUCK-A,-17.987039152468984


Databricks visualization. Run in Databricks to view.